# Experiment 4: Unit ball geometry and isometry groups

The residual symmetry of a regularizer reduces to a geometric question: *which linear maps send the $\ell_p$ unit ball to itself?* The answer depends entirely on the shape of the ball.

- The $\ell_2$ ball is a **sphere**: any rotation or reflection preserves it, giving the orthogonal group $O_d(\mathbb{R})$.
- The $\ell_1$ ball is a **cross-polytope** (diamond): only signed permutations of the coordinate axes preserve it, giving the hyperoctahedral group $B_d = \mathbb{Z}_2^d \rtimes S_d$.

This notebook visualizes these geometric facts and connects them to the sparsity-inducing property of $\ell_1$ regularization.


In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d import art3d

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'figure.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
})

np.random.seed(42)
print("Setup complete.")


Setup complete.


## 1. $\ell_p$ unit balls in 2D

The unit ball $\{x \in \mathbb{R}^2 : \|x\|_p \leq 1\}$ changes shape dramatically with $p$. As $p$ decreases from $\infty$ to $1$, the ball develops sharper corners along the coordinate axes. These corners are what make $\ell_1$ regularization produce sparse solutions.


In [2]:
def lp_ball_2d(p, n_points=1000):
    # Generate boundary of the Lp unit ball in 2D
    if p == np.inf:
        # Square
        t = np.linspace(0, 2*np.pi, n_points)
        x = np.sign(np.cos(t)) * np.minimum(np.abs(1/np.cos(t)), 1.0)
        y = np.sign(np.sin(t)) * np.minimum(np.abs(1/np.sin(t)), 1.0)
        # Cleaner: just draw the square
        x = np.array([1, 1, -1, -1, 1])
        y = np.array([1, -1, -1, 1, 1])
        return x, y
    
    theta = np.linspace(0, 2*np.pi, n_points)
    x = np.cos(theta)
    y = np.sin(theta)
    # Scale to Lp unit ball boundary
    r = (np.abs(np.cos(theta))**p + np.abs(np.sin(theta))**p)**(1/p)
    x = x / r
    y = y / r
    return x, y

fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
p_values = [0.5, 1, 2, 4, np.inf]
p_labels = ['$p = 0.5$', '$p = 1$', '$p = 2$', '$p = 4$', '$p = \\infty$']
colors = ['#E74C3C', '#E67E22', '#2E86C1', '#27AE60', '#8E44AD']

for ax, p, label, color in zip(axes, p_values, p_labels, colors):
    if p == 0.5:
        # Lp for p < 1 is not convex; show the "ball" anyway
        theta = np.linspace(0, 2*np.pi, 2000)
        cos_theta, sin_theta = np.cos(theta), np.sin(theta)
        r = (np.abs(cos_theta)**p + np.abs(sin_theta)**p)**(1/p)
        x, y = cos_theta/r, sin_theta/r
    else:
        x, y = lp_ball_2d(p)
    
    ax.fill(x, y, alpha=0.2, color=color)
    ax.plot(x, y, color=color, linewidth=2)
    ax.set_xlim(-1.6, 1.6)
    ax.set_ylim(-1.6, 1.6)
    ax.set_aspect('equal')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_title(label, fontsize=13)
    ax.grid(True, alpha=0.2)

plt.suptitle('$\\ell_p$ unit balls in $\\mathbb{R}^2$', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('Ex4_fig_lp_balls_2d.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print("Saved: Ex4_fig_lp_balls_2d.png")


/var/folders/xh/h88wz4193gzg53_x42zm8ngm0000gn/T/ipykernel_22723/1565404090.py:7: RuntimeWarning: divide by zero encountered in divide
  y = np.sign(np.sin(t)) * np.minimum(np.abs(1/np.sin(t)), 1.0)


Saved: Ex4_fig_lp_balls_2d.png


/var/folders/xh/h88wz4193gzg53_x42zm8ngm0000gn/T/ipykernel_22723/1565404090.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Isometry groups: applying transformations to the balls

The key insight: the **shape** of the unit ball determines its **isometry group**, and that isometry group is the residual symmetry of the corresponding regularization.

We demonstrate this by applying random transformations and checking whether the ball is preserved:
- **Rotation** (element of $O(2)$): preserves $\ell_2$ ball, distorts $\ell_1$ ball
- **Signed permutation** (element of $B_2$): preserves both $\ell_1$ and $\ell_2$ balls
- **General invertible**: distorts both


In [3]:
def rotation_matrix(theta):
    return np.array([[np.cos(theta), -np.sin(theta)],
                     [np.sin(theta),  np.cos(theta)]])

def random_signed_perm_2d():
    perm = np.random.permutation(2)
    signs = np.random.choice([-1, 1], size=2)
    M = np.zeros((2, 2))
    M[np.arange(2), perm] = signs
    return M

# Generate ball boundaries
theta_fine = np.linspace(0, 2*np.pi, 500)
l2_pts = np.column_stack([np.cos(theta_fine), np.sin(theta_fine)])

cos_theta, sin_theta = np.cos(theta_fine), np.sin(theta_fine)
r1 = np.abs(cos_theta) + np.abs(sin_theta)
l1_pts = np.column_stack([cos_theta/r1, sin_theta/r1])

# Apply transformations
angle = np.pi / 5  # 36 degrees
R = rotation_matrix(angle)
SP = random_signed_perm_2d()

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

for row, (ball_pts, ball_name, color) in enumerate([
    (l2_pts, '$\\ell_2$ ball (sphere)', '#2E86C1'),
    (l1_pts, '$\\ell_1$ ball (diamond)', '#E67E22')
]):
    for col, (M, transform_name) in enumerate([
        (R, f'Rotation by {int(np.degrees(angle))}°'),
        (SP, 'Signed permutation'),
        (np.array([[1.3, 0.4], [-0.2, 0.8]]), 'General invertible')
    ]):
        ax = axes[row, col]
        
        # Original
        ax.fill(ball_pts[:, 0], ball_pts[:, 1], alpha=0.15, color=color)
        ax.plot(ball_pts[:, 0], ball_pts[:, 1], color=color, linewidth=2, label='Original')
        
        # Transformed
        transformed = ball_pts @ M.T
        ax.fill(transformed[:, 0], transformed[:, 1], alpha=0.15, color='#E74C3C')
        ax.plot(transformed[:, 0], transformed[:, 1], color='#E74C3C', linewidth=2, 
                linestyle='--', label='Transformed')
        
        ax.set_xlim(-1.8, 1.8)
        ax.set_ylim(-1.8, 1.8)
        ax.set_aspect('equal')
        ax.axhline(0, color='gray', linewidth=0.5)
        ax.axvline(0, color='gray', linewidth=0.5)
        ax.grid(True, alpha=0.2)
        
        if row == 0:
            ax.set_title(transform_name, fontsize=12, fontweight='bold')
        if col == 0:
            ax.set_ylabel(ball_name, fontsize=12)
        
        # Check if preserved
        # Compute max deviation from unit ball
        if ball_name.startswith('$\\ell_2'):
            norms_orig = np.linalg.norm(ball_pts, axis=1)
            norms_trans = np.linalg.norm(transformed, axis=1)
        else:
            norms_orig = np.abs(ball_pts).sum(axis=1)
            norms_trans = np.abs(transformed).sum(axis=1)
        
        preserved = np.allclose(norms_trans, norms_orig, atol=1e-6)
        status = '✓ Preserved' if preserved else '✗ Distorted'
        status_color = '#27AE60' if preserved else '#E74C3C'
        ax.text(0.02, 0.98, status, transform=ax.transAxes, fontsize=11,
                verticalalignment='top', color=status_color, fontweight='bold')
        
        if row == 0 and col == 0:
            ax.legend(fontsize=9, loc='lower right')

plt.suptitle('Applying transformations to $\\ell_p$ unit balls', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('Ex4_fig_ball_transformations.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print("Saved: Ex4_fig_ball_transformations.png")


/var/folders/xh/h88wz4193gzg53_x42zm8ngm0000gn/T/ipykernel_22723/3413362683.py:80: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) DejaVu Serif.
  plt.tight_layout()
/var/folders/xh/h88wz4193gzg53_x42zm8ngm0000gn/T/ipykernel_22723/3413362683.py:80: UserWarning: Glyph 10007 (\N{BALLOT X}) missing from font(s) DejaVu Serif.
  plt.tight_layout()
/var/folders/xh/h88wz4193gzg53_x42zm8ngm0000gn/T/ipykernel_22723/3413362683.py:81: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) DejaVu Serif.
  plt.savefig('Ex4_fig_ball_transformations.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')


/var/folders/xh/h88wz4193gzg53_x42zm8ngm0000gn/T/ipykernel_22723/3413362683.py:81: UserWarning: Glyph 10007 (\N{BALLOT X}) missing from font(s) DejaVu Serif.
  plt.savefig('Ex4_fig_ball_transformations.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')


Saved: Ex4_fig_ball_transformations.png


/var/folders/xh/h88wz4193gzg53_x42zm8ngm0000gn/T/ipykernel_22723/3413362683.py:82: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Why $\ell_1$ induces sparsity: the classic visualization

When we minimize a smooth loss function subject to an $\ell_p$ constraint, the solution lies where the loss contours first touch the constraint set. For $\ell_2$, the constraint set is a circle — the contact point can be anywhere on the smooth boundary, with no preference for the coordinate axes.

For $\ell_1$, the constraint set is a cross-polytope (diamond in 2D) with vertices on the coordinate axes and flat faces between them. The solution lands on some face of this polytope, and the dimension of that face determines the sparsity: a solution on a $k$-dimensional face generically has $d - k - 1$ zero coordinates.

- On a $(d-1)$-dimensional facet: 0 zero coordinates (no sparsity)
- On a $(d-2)$-dimensional face: 1 zero coordinate
- On a 1-dimensional edge: $d-2$ zero coordinates
- On a 0-dimensional vertex: $d-1$ zero coordinates (maximally sparse)

The $(d-1)$-dimensional facets — the only faces giving *no* sparsity — have 1-dimensional normal cones (single rays on $S^{d-1}$), which form a measure-zero set of gradient directions. So for a generic loss function, the solution avoids the full-dimensional facets and lands on a lower-dimensional face, giving at least some zero coordinates.

In 2D this effect is weak: the diamond has 4 edges (1-dimensional, giving no sparsity) and 4 vertices (0-dimensional, giving 1 zero coordinate). A generic gradient direction hits an edge, not a vertex, so 2D $\ell_1$ sparsity is actually the non-generic case. But in high dimensions the picture reverses: in $\mathbb{R}^{100}$, even landing on a 50-dimensional face gives ~49 zero coordinates. The generic solution has substantial sparsity; the vertex (maximum sparsity) is just the extreme case.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Elliptical loss contours (representing a quadratic loss with non-axis-aligned minimum)
# Center placed so that contours clearly first touch the L1 diamond at a vertex,
# making the sparsity argument visually obvious.
center = np.array([0.1, 1.8])
# Covariance for elliptical contours (elongated in x, so the ellipse wraps the vertex)
angle_ell = np.pi / 12
R_ell = rotation_matrix(angle_ell)
D_ell = np.diag([3.0, 1.0])
Sigma_inv = R_ell @ D_ell @ R_ell.T

xx, yy = np.meshgrid(np.linspace(-2, 2.5, 300), np.linspace(-2, 3, 300))
pts = np.column_stack([xx.ravel(), yy.ravel()]) - center
loss_vals = (pts @ Sigma_inv * pts).sum(axis=1).reshape(xx.shape)

for ax_idx, (ax, p, ball_name, color) in enumerate([
    (axes[0], 2, '$\\ell_2$ constraint', '#2E86C1'),
    (axes[1], 1, '$\\ell_1$ constraint', '#E67E22')
]):
    # Loss contours
    levels = np.linspace(0.1, 8, 15)
    ax.contour(xx, yy, loss_vals, levels=levels, colors='gray', alpha=0.4, linewidths=0.8)
    
    # Constraint set
    if p == 2:
        theta_c = np.linspace(0, 2*np.pi, 500)
        radius = 0.9
        cx, cy = radius * np.cos(theta_c), radius * np.sin(theta_c)
    else:
        radius = 0.9
        cx = np.array([radius, 0, -radius, 0, radius])
        cy = np.array([0, radius, 0, -radius, 0])
    
    ax.fill(cx, cy, alpha=0.2, color=color)
    ax.plot(cx, cy, color=color, linewidth=2.5, label=ball_name)
    
    # Find approximate tangent point (where loss contour touches constraint)
    if p == 2:
        # For L2: project center onto circle
        direction = -center / np.linalg.norm(center)
        # Actually find the point on the circle closest to the loss minimum
        # by minimizing loss on the circle
        theta_search = np.linspace(0, 2*np.pi, 10000)
        circle_pts = radius * np.column_stack([np.cos(theta_search), np.sin(theta_search)])
        diffs = circle_pts - center
        losses_on_circle = (diffs @ Sigma_inv * diffs).sum(axis=1)
        best_idx = np.argmin(losses_on_circle)
        touch_pt = circle_pts[best_idx]
    else:
        # For L1: find the point on the diamond boundary closest to loss minimum
        # Check all edges by parametrizing each one
        verts = radius * np.array([[1, 0], [0, 1], [-1, 0], [0, -1]])
        best_loss_l1 = np.inf
        touch_pt = verts[0]
        for edge_i in range(4):
            v1, v2 = verts[edge_i], verts[(edge_i + 1) % 4]
            for alpha in np.linspace(0, 1, 2000):
                pt = (1 - alpha) * v1 + alpha * v2
                d = pt - center
                l = d @ Sigma_inv @ d
                if l < best_loss_l1:
                    best_loss_l1 = l
                    touch_pt = pt.copy()
    
    ax.plot(*touch_pt, 'o', color='#E74C3C', markersize=10, zorder=5)
    ax.annotate(f'Solution\n({touch_pt[0]:.2f}, {touch_pt[1]:.2f})', 
                xy=touch_pt, xytext=(touch_pt[0]+0.5, touch_pt[1]+0.6),
                fontsize=10, color='#E74C3C',
                arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=1.5))
    
    # Mark the unconstrained minimum
    ax.plot(*center, '*', color='gray', markersize=12, zorder=5)
    ax.annotate('Unconstrained\nminimum', xy=center, xytext=(center[0]+0.3, center[1]-0.7),
                fontsize=9, color='gray',
                arrowprops=dict(arrowstyle='->', color='gray', lw=1))
    
    ax.set_xlim(-1.8, 2.5)
    ax.set_ylim(-1.8, 2.8)
    ax.set_aspect('equal')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=11, loc='lower left')
    
    if ax_idx == 0:
        ax.set_title('$\\ell_2$: solution on smooth boundary\n(no sparsity)', fontsize=12)
    else:
        ax.set_title('$\\ell_1$: solution at vertex\n(sparse — one coordinate is zero)', fontsize=12)

plt.suptitle('Why $\\ell_1$ regularization induces sparsity', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('Ex4_fig_sparsity_geometry.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print("Saved: Ex4_fig_sparsity_geometry.png")


## 4. 3D unit balls

The same story extends to higher dimensions. In 3D, the $\ell_2$ ball is a sphere (continuous rotational symmetry), while the $\ell_1$ ball is an octahedron (only 48 symmetries: the hyperoctahedral group $B_3 = \mathbb{Z}_2^3 \rtimes S_3$, with $|B_3| = 2^3 \cdot 3! = 48$).


In [5]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

fig = plt.figure(figsize=(14, 6))

# L2 ball (sphere)
ax1 = fig.add_subplot(121, projection='3d')
u = np.linspace(0, 2*np.pi, 50)
v = np.linspace(0, np.pi, 50)
x_s = np.outer(np.cos(u), np.sin(v))
y_s = np.outer(np.sin(u), np.sin(v))
z_s = np.outer(np.ones_like(u), np.cos(v))
ax1.plot_surface(x_s, y_s, z_s, alpha=0.3, color='#2E86C1', edgecolor='#2E86C1', linewidth=0.1)
ax1.set_title('$\\ell_2$ ball: sphere\nIsometry group: $O(3)$ (infinite)', fontsize=12)
ax1.set_xlim(-1.2, 1.2); ax1.set_ylim(-1.2, 1.2); ax1.set_zlim(-1.2, 1.2)
ax1.set_xlabel('$x_1$'); ax1.set_ylabel('$x_2$'); ax1.set_zlabel('$x_3$')

# L1 ball (octahedron)
ax2 = fig.add_subplot(122, projection='3d')
vertices = np.array([[1,0,0],[-1,0,0],[0,1,0],[0,-1,0],[0,0,1],[0,0,-1]], dtype=float)
faces = [
    [0, 2, 4], [0, 4, 3], [0, 3, 5], [0, 5, 2],
    [1, 2, 4], [1, 4, 3], [1, 3, 5], [1, 5, 2],
]
poly = Poly3DCollection([vertices[f] for f in faces], alpha=0.3, 
                         facecolor='#E67E22', edgecolor='#E67E22', linewidth=0.8)
ax2.add_collection3d(poly)
# Mark vertices
ax2.scatter(*vertices.T, color='#E74C3C', s=40, zorder=5)
ax2.set_title('$\\ell_1$ ball: octahedron\nIsometry group: $B_3$ (48 elements)', fontsize=12)
ax2.set_xlim(-1.2, 1.2); ax2.set_ylim(-1.2, 1.2); ax2.set_zlim(-1.2, 1.2)
ax2.set_xlabel('$x_1$'); ax2.set_ylabel('$x_2$'); ax2.set_zlabel('$x_3$')

plt.tight_layout()
plt.savefig('Ex4_fig_3d_balls.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print("Saved: Ex4_fig_3d_balls.png")


Saved: Ex4_fig_3d_balls.png


/var/folders/xh/h88wz4193gzg53_x42zm8ngm0000gn/T/ipykernel_22723/3583902205.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

The geometry of the regularization norm's unit ball completely determines the residual symmetry group:

| Norm | Unit ball shape | Isometry group | Size | Sparsity? |
|---|---|---|---|---|
| $\ell_2$ | Sphere | $O_d(\mathbb{R})$ | Infinite (continuous) | No |
| $\ell_1$ | Cross-polytope | $B_d = \mathbb{Z}_2^d \rtimes S_d$ | $2^d \cdot d!$ (finite) | Yes |
| $\ell_\infty$ | Hypercube | $B_d$ | $2^d \cdot d!$ (finite) | No (but dual to $\ell_1$) |

The Banach–Lamperti theorem makes this rigorous: for $p \neq 2$, the only $\ell_p$ isometries are signed permutations. The sphere is special — it has continuous rotational symmetry that no other $\ell_p$ ball possesses. This is why $\ell_2$ regularization preserves rotational freedom (no preferred basis) while $\ell_1$ regularization selects the coordinate axes as special directions.

**Remark: isometry group vs weight sparsity.** The elements of $O_d(\mathbb{R})$ are orthogonal matrices, and $\ell_2$ regularization selects orthogonal weight matrices — the group's matrix representation directly describes the structure of the preferred weights. One might expect the same for $\ell_1$: the elements of $B_d$ are signed permutation matrices, which are themselves sparse ($d$ nonzero entries out of $d^2$). But this is not why $\ell_1$ regularization produces sparse weights. The sparsity of the *group elements* and the sparsity of the *learned weights* have different origins. The group $B_d$ tells you which reparametrizations the $\ell_1$ penalty cannot distinguish (its residual symmetry). The sparsity of the learned weights comes from the *geometry of the unit ball*: the cross-polytope has vertices on the coordinate axes, and constrained optimization generically lands at these vertices (section 3 above). The group structure and the ball geometry are related — $B_d$ is the isometry group *because* the cross-polytope has that shape — but the causal chain to weight sparsity runs through the ball geometry, not through the matrix representation of the group.
